# A-S00 SegResNet data setup and QA

This notebook verifies the frozen TopAneu 2026 MRA SegResNet
location-segmentation dataset.

Contract:

- 215 train, 46 validation, 46 held-out test
- development manifest contains only train and validation
- seed 20260828; split regeneration prohibited
- images + location masks + location JSON only
- type masks and nnU-Net Dataset501/502 are excluded
- credentials, authentication URLs and patient data are never committed


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path("/home/jupyter-mirae8/Aneu_MRA/model")
DATA_ROOT = Path("/home/jupyter-mirae8/Aneu_MRA/data")

os.environ["BRAINON_DATA_ROOT"] = str(DATA_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

print("Repository:", REPO_ROOT)
print("Data root :", DATA_ROOT)


In [ ]:
from brainon_ai.models.aneurysm.segresnet.config import load_config
from brainon_ai.models.aneurysm.segresnet.data import load_manifest

config = load_config()
manifest = load_manifest(config)

counts = manifest.groupby("split").size().to_dict()

assert counts == {"train": 215, "validation": 46}
assert "test" not in set(manifest["split"])
assert config["data"]["use_type_masks"] is False

print("Cases :", len(manifest))
print("Splits:", counts)
print("PASS: frozen A-S00 development manifest")


In [ ]:
from brainon_ai.models.aneurysm.segresnet.checkpointing import sha256_file
from brainon_ai.models.aneurysm.segresnet.config import resolve_repository_path

common_manifest = resolve_repository_path(
    config["data"]["common_manifest_path"]
)
split_info = resolve_repository_path(
    config["data"]["split_info_path"]
)

assert sha256_file(common_manifest) == config["data"]["common_manifest_sha256"]
assert sha256_file(split_info) == config["data"]["split_info_sha256"]

print("Common manifest SHA-256:", sha256_file(common_manifest))
print("Split info SHA-256     :", sha256_file(split_info))
print("PASS: frozen metadata hashes")
